# Thí nghiệm 3: Đánh giá mô hình (Model Selection)
## Mục tiêu: So sánh hiệu suất của Logistic Regression và Decision Tree bằng Cross-Validation

### 3.1 Mục tiêu & Kì vọng kết quả

**Bối cảnh thực tế:**
- Một ngân hàng muốn dự đoán liệu khách hàng có đăng ký khoản tiền gửi có kỳ hạn (term deposit) hay không dựa trên các thông tin cá nhân và lịch sử tương tác.
- Chúng ta cần lựa chọn giữa hai mô hình: **Logistic Regression** (Mô hình tuyến tính, đơn giản) và **Decision Tree** (Mô hình phi tuyến, dễ bị overfit).

**Câu hỏi chính:** "Mô hình nào thực sự tốt hơn và ổn định hơn trên dữ liệu chưa biết?"

**Kì vọng kết quả:**
1. **Single Split (Train/Test split)**: Kết quả Accuracy sẽ biến động mạnh khi thay đổi `random_state`. Có thể xảy ra trường hợp mô hình A tốt hơn B ở lần chia này, nhưng lại kém hơn ở lần chia khác.
2. **Cross-Validation**: Cung cấp một cái nhìn toàn diện hơn về hiệu suất thông qua giá trị trung bình và phương sai của các fold. Chúng ta kì vọng thấy được mô hình nào thực sự ổn định hơn.
3. **Kết luận kì vọng**: Resampling (CV) giúp ta tránh việc đưa ra quyết định sai lầm khi lựa chọn mô hình chỉ dựa trên một lần chia dữ liệu may rủi.

### Mô tả Dataset: Bank Marketing dataset
- **Nguồn**: UCI Machine Learning Repository.
- **Đặc điểm**: Dữ liệu bao gồm các thông tin như tuổi, nghề nghiệp, tình trạng hôn nhân, số dư tài khoản, và kết quả của các chiến dịch marketing trước đó.

| Cột | Ý nghĩa | Ghi chú |
|:---|:---|:---|
| **age** | Tuổi khách hàng | Biến định lượng |
| **job** | Nghề nghiệp | admin, blue-collar, technician, etc. |
| **marital** | Tình trạng hôn nhân | married, single, divorced |
| **education** | Học vấn | primary, secondary, tertiary, unknown |
| **balance** | Số dư tài khoản trung bình hàng năm | Đơn vị: Euro |
| **housing** | Có vay thế chấp nhà không? | yes/no |
| **loan** | Có vay cá nhân không? | yes/no |
| **contact** | Hình thức liên lạc | cellular, telephone |
| **duration** | Thời lượng cuộc gọi cuối cùng | Đơn vị: giây |
| **campaign** | Số lần liên lạc trong chiến dịch này | Bao gồm cả lần cuối |
| **pdays** | Số ngày trôi qua kể từ lần liên lạc trước | -1 nghĩa là chưa liên lạc |
| **deposit** | Khách hàng có đăng ký tiền gửi không? | **Biến mục tiêu (yes/no)** |

- **Tại sao chọn dataset này?**:
    - Dữ liệu có sự mất cân bằng nhẹ giữa các lớp (nhãn 'yes' và 'no').
    - Có sự kết hợp giữa biến định tính và định lượng, tạo ra sự phức tạp đủ để thấy sự khác biệt giữa các mô hình.
    - Phù hợp để minh họa việc kết quả Accuracy có thể bị nhiễu nếu chỉ dùng một lần split.

### Các chỉ số đo lường (Metrics):

Chúng ta sử dụng các chỉ số sau để đánh giá độ ổn định và hiệu suất mô hình:
1. **Accuracy Mean (Độ chính xác trung bình)**: Hiệu suất trung bình của mô hình trên các fold (trong CV) hoặc qua các lần split.
2. **Accuracy Variance/Std (Độ biến thiên)**: Đo lường mức độ nhạy cảm của mô hình với việc thay đổi dữ liệu huấn luyện. Phương sai cao đồng nghĩa với việc mô hình kém ổn định.
3. **Generalization Error Estimate**: Ước lượng sai số khi áp dụng mô hình vào dữ liệu thực tế ngoài tập huấn luyện.

In [1]:
# Bước 1: Tải dataset
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data')
os.makedirs(data_dir, exist_ok=True)
path = os.path.join(data_dir, 'bank.csv')

if not os.path.exists(path):
    url = "https://raw.githubusercontent.com/janiobachmann/Bank-Marketing-Dataset/master/bank.csv"
    print(f"Downloading dataset from {url}...")
    df = pd.read_csv(url)
    df.to_csv(path, index=False)
    print(f"Saved to {path}")
else:
    df = pd.read_csv(path)
    print(f"File already exists: {path}")

df.head()

HTTPError: HTTP Error 404: Not Found

In [ ]:
# Tiền xử lý dữ liệu đơn giản
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Copy để tránh ảnh hưởng df gốc
df_proc = df.copy()

# Encode các biến định tính
le = LabelEncoder()
categorical_cols = df_proc.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_proc[col] = le.fit_transform(df_proc[col])

X = df_proc.drop('deposit', axis=1)
y = df_proc['deposit']

# Scale dữ liệu cho Logistic Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Kích thước dữ liệu: {X.shape}")

### 3.2 Tiến hành thí nghiệm

#### Bước 1: Vấn đề với Single Split
Chúng ta sẽ chạy thí nghiệm Single Split 10 lần với các `random_state` khác nhau để xem kết quả Accuracy biến động như thế nào.

In [ ]:
results_single = []
seeds = range(10, 20)

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=seed)
    
    # Logistic Regression
    lr = LogisticRegression()
    lr.fit(X_train, y_train)
    acc_lr = lr.score(X_test, y_test)
    
    # Decision Tree
    dt = DecisionTreeClassifier(max_depth=5)
    dt.fit(X_train, y_train)
    acc_dt = dt.score(X_test, y_test)
    
    results_single.append({'seed': seed, 'Logistic Regression': acc_lr, 'Decision Tree': acc_dt})

df_single = pd.DataFrame(results_single)
print("Kết quả Accuracy qua các lần split khác nhau:")
print(df_single)

#### Bước 2: Sự ổn định của Cross-Validation
Bây giờ chúng ta sử dụng K-Fold Cross Validation (K=10) để xem phân phối Accuracy ổn định hơn.

In [ ]:
lr_cv = cross_val_score(LogisticRegression(), X_scaled, y, cv=10)
dt_cv = cross_val_score(DecisionTreeClassifier(max_depth=5), X_scaled, y, cv=10)

print(f"Logistic Regression CV Accuracy: {lr_cv.mean():.4f} (+/- {lr_cv.std():.4f})")
print(f"Decision Tree CV Accuracy: {dt_cv.mean():.4f} (+/- {dt_cv.std():.4f})")

### Bước 3: Vẽ biểu đồ minh họa

In [ ]:
plt.figure(figsize=(14, 6))

# Biểu đồ 1: Biến động của Single Split(ex3_single_split)
plt.subplot(1, 2, 1)
plt.plot(df_single['seed'], df_single['Logistic Regression'], marker='o', label='Logistic Regression')
plt.plot(df_single['seed'], df_single['Decision Tree'], marker='s', label='Decision Tree')
plt.title("Accuracy của Single Split qua các Seed khác nhau")
plt.xlabel("Seed")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

# Biểu đồ 2: So sánh phân phối qua Boxplot (ex3_kfold)
plt.subplot(1, 2, 2)
cv_data = pd.DataFrame({
    'Logistic Regression': lr_cv,
    'Decision Tree': dt_cv
})
sns.boxplot(data=cv_data)
plt.title("Phân phối Accuracy qua 10-Fold Cross Validation")
plt.ylabel("Accuracy")

plt.tight_layout()
plt.show()

### 3.3 Nhận xét thí nghiệm

**Hiện tượng trong plot:**
- Ở biểu đồ bên trái (Single Split), ta thấy đường Accuracy của hai mô hình đan xen nhau. Tại một số Seed, Logistic Regression tốt hơn, nhưng ở các Seed khác, Decision Tree lại vọt lên dẫn trước. Khoảng dao động của Accuracy khá lớn (có thể lên tới 2-3%).
- Ở biểu đồ bên phải (Boxplot CV), ta thấy được bức tranh tổng thể. Dải Accuracy của Decision Tree thường có trung vị cao hơn hoặc thấp hơn một cách rõ ràng, và độ trải rộng (whisker) cho thấy mô hình nào có rủi ro cao hơn trên các tập dữ liệu con khác nhau.

**Đặc điểm tập dữ liệu:**
- Tập dữ liệu Bank Marketing có các đặc trưng tương tác phi tuyến tính phức tạp. Điều này giải thích tại sao Decision Tree (nếu được giới hạn chiều sâu phù hợp) thường có tiềm năng đạt accuracy cao hơn Logistic Regression.
- Tuy nhiên, do dữ liệu có tính nhiễu, việc chỉ split một lần dễ khiến mô hình học được các mẫu đặc thù của tập train đó mà không đại diện cho toàn bộ quần thể.

**Kết luận cuối cùng:**
- **Cross-Validation là bắt buộc**: Nếu chỉ dựa vào một lần split ngẫu nhiên, ta rất dễ đưa ra kết luận sai về việc mô hình nào tốt hơn (ví dụ: chọn Logistic Regression chỉ vì nó may mắn đạt accuracy cao ở seed 12).
- **Sự ổn định**: CV giúp ta ước lượng được "sai số tổng quát hóa" một cách khách quan. Trong thí nghiệm này, mặc dù Accuracy có thể biến động, nhưng trung bình của CV cho thấy Decision Tree có xu hướng hoạt động tốt hơn trên dữ liệu này, giúp ta tự tin hơn khi lựa chọn nó để triển khai thực tế.